# Tome 15, Appendix H: Points, Distances and Families of Sets (Python companion)

Exercises first, solutions at the end. We check the Moser spindle and Isbell's hexagon colouring, count unit distances in two constructions, test the
eight points of Proposition 7.5 and the value $f(5)=9$ on random points, and run Gilmer's entropy argument (Theorem 7.8) on random
union-closed families. We use `numpy` and `matplotlib`.

*Before each solution, predict what the output will look like.*

In [1]:
import itertools, math
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(8)

## Exercises

**P8.1 (★) The Moser spindle.** Build the seven points of Theorem 7.1, list all pairs at distance $1$ (up to $10^{-9}$), and check by brute force
that the graph is not $3$-colourable but becomes $3$-colourable when any edge is removed (Exercise 7.2).

**P8.2 (★★) Hexagons.** Implement Isbell's colouring with side $s$: find the hexagon containing a point, then its colour. Test it on a million random
pairs of points at distance $1$. Then compute the exact distance between two hexagons of the same colour, and the true range of $s$ for which the
colouring works.

**P8.3 (★) Unit distances.** Count the unit distances in the cube construction of Exercise 7.4 for $k\le10$, and in a square grid where
the unit is the distance $\sqrt r$ realised most often. Compare with $\frac n2\log_2n$ and with the upper bound $4n^{4/3}+n$ of Theorem 7.2.

**P8.4 (★) Pentagons.** Check that the eight points of Proposition 7.5 are in general position with no five in convex position. Then draw
$9$ random points many times, and $8$ random points many times: how often is there no convex pentagon?

**P8.5 (★★) Union-closed families.** Generate random union-closed families as unions of random generators. For each, compute the largest frequency,
and the terms of the proof of Theorem 7.8: $H(A)$, $H(A\cup B)$, $\sum_i\mathbb E\,h(U_iV_i)$ and $c\sum_i\mathbb E[U_i]\,\mathbb E\,h(U_i)$.
Also check Lemma 7.9 and Boppana's inequality on a grid.

## Solutions

### P8.1
A $3$-colouring search over $3^7=2187$ assignments is instant.

In [2]:
th = 2 * math.asin(1 / (2 * math.sqrt(3)))
def rot(p, a): return (p[0] * math.cos(a) - p[1] * math.sin(a), p[0] * math.sin(a) + p[1] * math.cos(a))
A = (0.0, 0.0); B = (math.cos(math.pi/3 + 0.1), math.sin(math.pi/3 + 0.1)); C_ = rot(B, math.pi/3)
D = (B[0] + C_[0], B[1] + C_[1])
E, F, G = rot(B, -th), rot(C_, -th), rot(D, -th)
P = dict(A=A, B=B, C=C_, D=D, E=E, F=F, G=G)
edges = [(u, v) for u, v in itertools.combinations(P, 2) if abs(math.dist(P[u], P[v]) - 1) < 1e-9]
print(len(edges), 'unit distances:', ' '.join(u + v for u, v in edges))
def colourable(E, k=3):
    V = list(P)
    return any(all(c[V.index(u)] != c[V.index(v)] for u, v in E) for c in itertools.product(range(k), repeat=len(V)))
print('3-colourable:', colourable(edges))
print('3-colourable after removing each edge:', all(colourable([e for e in edges if e != f]) for f in edges))

11 unit distances: AB AC AE AF BC BD CD DG EF EG FG
3-colourable: False
3-colourable after removing each edge: True


Exactly the eleven edges of the text, and the spindle is $4$-critical. (The construction starts from an arbitrary tilt $0.1$ of the first rhombus;
any tilt works, since the spindle is rigid only up to rotation.)

### P8.2
Centres $ia_1+ja_2$ with $a_1=\sqrt3s(1,0)$, $a_2=\sqrt3s(\frac12,\frac{\sqrt3}2)$ (pointy-top hexagons). The hexagon containing a point is the one with
the nearest centre (Voronoi cells of the triangular lattice are exactly these hexagons), and we find it by checking the lattice points around the
rounded coordinates.

In [3]:
def colour(pts, s):
    a1 = np.array([math.sqrt(3) * s, 0.0]); a2 = np.array([math.sqrt(3) * s / 2, 1.5 * s])
    M = np.linalg.inv(np.column_stack([a1, a2]))
    ij = pts @ M.T
    base = np.floor(ij).astype(int)
    best = None; bestd = None
    for di in (-1, 0, 1, 2):
        for dj in (-1, 0, 1, 2):
            cand = base + np.array([di, dj])
            cen = cand[:, :1] * a1 + cand[:, 1:] * a2
            d = np.linalg.norm(pts - cen, axis=1)
            if best is None:
                best, bestd = cand.copy(), d
            else:
                m = d < bestd; best[m] = cand[m]; bestd[m] = d[m]
    return (best[:, 0] + 3 * best[:, 1]) % 7
for s in (0.40, 0.45, 0.49):
    N = 10**6
    p = rng.uniform(-20, 20, size=(N, 2)); t = rng.uniform(0, 2 * np.pi, N)
    q = p + np.column_stack([np.cos(t), np.sin(t)])
    print('s = %.2f: monochromatic unit pairs among 10^6 random ones: %d' % (s, np.sum(colour(p, s) == colour(q, s))))

def hexagon(c, s):
    return [c + s * np.array([math.cos(math.pi/6 + k * math.pi/3), math.sin(math.pi/6 + k * math.pi/3)]) for k in range(6)]
def seg_dist(p, a, b):
    t = np.clip(np.dot(p - a, b - a) / np.dot(b - a, b - a), 0, 1); return np.linalg.norm(p - a - t * (b - a))
def poly_dist(P1, P2):
    return min(min(seg_dist(p, P2[k], P2[(k+1) % 6]) for p in P1 for k in range(6)),
               min(seg_dist(p, P1[k], P1[(k+1) % 6]) for p in P2 for k in range(6)))
s = 1.0
a1 = np.array([math.sqrt(3) * s, 0.0]); a2 = np.array([math.sqrt(3) * s / 2, 1.5 * s])
same = [(i, j) for i in range(-6, 7) for j in range(-6, 7) if (i, j) != (0, 0) and (i + 3 * j) % 7 == 0]
gap = min(poly_dist(hexagon(np.zeros(2), s), hexagon(i * a1 + j * a2, s)) for i, j in same)
print('closest same-colour centres: sqrt(21) s = %.4f s' % min(math.hypot(*(i * a1 + j * a2)) for i, j in same))
print('exact gap between same-colour hexagons = %.6f s  (sqrt(7) = %.6f, the bound sqrt(21)-2 = %.6f)' % (gap, math.sqrt(7), math.sqrt(21) - 2))
print('the colouring works for %.4f < s < 0.5' % (1 / gap))

s = 0.40: monochromatic unit pairs among 10^6 random ones: 0


s = 0.45: monochromatic unit pairs among 10^6 random ones: 0


s = 0.49: monochromatic unit pairs among 10^6 random ones: 0
closest same-colour centres: sqrt(21) s = 4.5826 s
exact gap between same-colour hexagons = 2.645751 s  (sqrt(7) = 2.645751, the bound sqrt(21)-2 = 2.582576)
the colouring works for 0.3780 < s < 0.5


No monochromatic unit pair in three million trials. The true gap between two same-coloured hexagons is $\sqrt7\,s$, larger than the bound
$(\sqrt{21}-2)s$ used in the proof, so the colouring works for $\frac1{\sqrt7}=0.378\ldots<s<\frac12$, not only for $s>0.3871$. The upper end
$s<\frac12$ is forced by the diameter of one hexagon.

### P8.3
For the grid, take the $m\times m$ grid and the squared distance $r$ with the most representations: the number of unit pairs is the number of pairs
of grid points at squared distance $r$, computed by convolution of the counts of difference vectors.

In [4]:
rows = []
for k in range(2, 11):
    n = 2**k
    th = rng.uniform(0, np.pi, k)
    V = np.column_stack([np.cos(th), np.sin(th)])
    pts = np.array([[(x >> i) & 1 for i in range(k)] for x in range(n)]) @ V
    D = np.linalg.norm(pts[:, None] - pts[None], axis=2)
    u = int((np.abs(D - 1) < 1e-9).sum() // 2)
    assert len({(round(a, 9), round(b, 9)) for a, b in pts}) == n
    rows.append(('cube', n, u))
for m in (5, 10, 20, 30, 40):
    n = m * m
    cnt = {}
    for dx in range(-(m-1), m):
        for dy in range(-(m-1), m):
            if (dx, dy) > (0, 0):
                r = dx * dx + dy * dy
                cnt[r] = cnt.get(r, 0) + (m - abs(dx)) * (m - abs(dy))
    r, u = max(cnt.items(), key=lambda kv: kv[1])
    rows.append(('grid r=%d' % r, n, u))
print('%-12s %6s %8s %10s %12s' % ('', 'n', 'u', '(n/2)lg n', '4n^(4/3)+n'))
for name, n, u in rows:
    print('%-12s %6d %8d %10.0f %12.0f' % (name, n, u, n / 2 * math.log2(n), 4 * n**(4/3) + n))

                  n        u  (n/2)lg n   4n^(4/3)+n
cube              4        4          4           29
cube              8       12         12           72
cube             16       32         32          177
cube             32       80         80          438
cube             64      192        192         1088
cube            128      448        448         2708
cube            256     1024       1024         6758
cube            512     2304       2304        16896
cube           1024     5120       5120        42309
grid r=5         25       48         58          317
grid r=5        100      288        332         1957
grid r=65       400     1744       1729        12189
grid r=65       900     4944       4416        35658
grid r=65      1600     9744       8515        76455


The cube construction gives exactly $\frac n2\log_2n$. The grid wins as $n$ grows, because the best squared distance $r$ (a product of many primes
$\equiv1\bmod4$) has many representations as a sum of two squares: this is Erd\H os's $n^{1+c/\log\log n}$, here still modest. Both are far below the
upper bound $4n^{4/3}+n$, and the gap between $n^{1+o(1)}$ and $n^{4/3}$ is the unit distance problem.

### P8.4
Five points are in convex position when none is inside the triangle of three others.

In [5]:
def orient(a, b, c): return (b[0]-a[0]) * (c[1]-a[1]) - (b[1]-a[1]) * (c[0]-a[0])
def in_tri(p, a, b, c):
    d1, d2, d3 = orient(a, b, p), orient(b, c, p), orient(c, a, p)
    return (d1 > 0 and d2 > 0 and d3 > 0) or (d1 < 0 and d2 < 0 and d3 < 0)
def convex5(S):
    return not any(in_tri(p, *t) for p in S for t in itertools.combinations([q for q in S if q is not p], 3))
def has_pentagon(pts):
    return any(convex5(S) for S in itertools.combinations(pts, 5))
r90 = lambda p: (-p[1], p[0])
Ps = [(6, 0)]; Qs = [(1, 2)]
for _ in range(3): Ps.append(r90(Ps[-1])); Qs.append(r90(Qs[-1]))
eight = Ps + Qs
print('collinear triples:', sum(orient(*t) == 0 for t in itertools.combinations(eight, 3)), ' convex pentagon:', has_pentagon(eight))
for N in (8, 9):
    trials = 3000
    none = sum(not has_pentagon([tuple(p) for p in rng.uniform(0, 1, (N, 2))]) for _ in range(trials))
    print('%d random points: no convex pentagon in %d of %d trials' % (N, none, trials))

collinear triples: 0  convex pentagon: False


8 random points: no convex pentagon in 4 of 3000 trials


9 random points: no convex pentagon in 0 of 3000 trials


The eight points pass: no three collinear, no convex pentagon. Random points tell the opposite story: $9$ random points never avoid a convex
pentagon (as $f(5)=9$ says they cannot), but neither do $8$ random points in practice. Configurations without a convex pentagon are rare and special,
which is why lower bounds for $f(n)$ need constructions, and why computer searches for $f(6)$ and beyond need clever pruning rather than sampling.

### P8.5
A family is a set of bitmasks; we close random generators under union. For the entropies, sets are bit vectors, and $U_i=P(A_i=0\mid A_{<i})$ is
computed by grouping the sets by their first $i-1$ bits.

In [6]:
def h(t):
    t = np.asarray(t, float); out = np.zeros_like(t)
    m = (t > 0) & (t < 1); out[m] = -t[m] * np.log(t[m]) - (1 - t[m]) * np.log(1 - t[m]); return out
def closure(gens):
    F = set(gens)
    while True:
        new = {a | b for a in F for b in F} - F
        if not new: return sorted(F)
        F |= new
def gilmer_terms(F, n):
    F = np.array(F); N = len(F)
    bits = (F[:, None] >> np.arange(n)) & 1
    HA = math.log(N)
    UB = np.array([a | b for a in F for b in F]); _, c = np.unique(UB, return_counts=True)
    HC = float(-(c / N**2 * np.log(c / N**2)).sum())
    S_uv = S_u = 0.0
    for i in range(n):
        prefix = F & ((1 << i) - 1)
        keys, inv, cnt = np.unique(prefix, return_inverse=True, return_counts=True)
        zeros = np.bincount(inv, weights=1 - bits[:, i])
        u = zeros / cnt; w = cnt / N
        S_uv += float((w[:, None] * w[None, :] * h(u[:, None] * u[None, :])).sum())
        S_u += float((w * u).sum() * (w * h(u)).sum())
    freq = bits.mean(axis=0).max()
    return HA, HC, S_uv, S_u, freq
c = 2 / (1 + math.log(2))
print('%4s %6s %7s %8s %9s %11s %6s' % ('n', '|F|', 'H(A)', 'H(AuB)', 'sum Eh(UV)', 'c sum EU Eh', 'maxfr'))
worst = 1
for trial in range(12):
    n = int(rng.integers(6, 11))
    gens = [int(x) for x in rng.integers(1, 2**n, size=int(rng.integers(3, 8)))]
    gens = [g & int(rng.integers(1, 2**n)) or g for g in gens]
    F = closure(gens + [0])
    HA, HC, Suv, Su, fr = gilmer_terms(F, n)
    worst = min(worst, fr)
    assert HC <= HA + 1e-9 and Suv <= HC + 1e-9 and c * Su <= Suv + 1e-9
    print('%4d %6d %7.3f %8.3f %9.3f %11.3f %6.3f' % (n, len(F), HA, HC, Suv, c * Su, fr))
x = np.linspace(0, 1, 1001); X, Y = np.meshgrid(x, x)
lhs = h(X * Y); rhs = X * h(Y) + Y * h(X)
print('min of h(xy) - rhs/(1+ln2) on the grid: %.2e' % (lhs - rhs / (1 + math.log(2))).min())
phi = (1 + 5**0.5) / 2
print('min of h(xy) - (phi/2) rhs on the grid: %.2e' % (lhs - phi / 2 * rhs).min())

   n    |F|    H(A)   H(AuB) sum Eh(UV) c sum EU Eh  maxfr
   9     16   2.773    2.366     2.177       1.541  0.750
   8      8   2.079    1.687     1.687       1.228  0.500
   8      8   2.079    1.687     1.687       1.228  0.500
   6      8   2.079    1.803     1.774       1.253  0.750
   7     24   3.178    2.677     2.433       1.674  0.792
  10     42   3.738    2.993     2.593       1.817  0.762
   8     16   2.773    2.337     2.320       1.663  0.750
   8     10   2.303    1.862     1.846       1.326  0.600
   9      7   1.946    1.644     1.633       1.166  0.714
  10      8   2.079    1.687     1.546       1.061  0.875
   7     22   3.091    2.543     2.424       1.677  0.864
   6      4   1.386    1.125     1.125       0.819  0.500
min of h(xy) - rhs/(1+ln2) on the grid: 0.00e+00
min of h(xy) - (phi/2) rhs on the grid: 0.00e+00


In every family the chain of inequalities of the proof holds: $c\sum\mathbb E[U]\,\mathbb E\,h(U)\le\sum\mathbb E\,h(UV)\le H(A\cup B)\le H(A)$. The
largest frequency is always well above $\frac12$ in random families: Frankl's conjecture is hard because of rare, structured families, not typical
ones. The two numerical checks of the pointwise inequalities come out nonnegative (Boppana's to rounding, since it has equality at $x=y=\frac1\varphi$).